In [28]:
import torch

# let's encode your example as integers for clarity:
#   a→0, m→1, c→2, x→3, n→4, d→5
t = torch.tensor([
    [0,0,0, 1,1,1, 2,2,2,2],
    [3,3,3, 4,4,4, 5,5,5,5],
])

# build your mask (==1 for 'm', ==4 for 'n')
mask = (t == 1) | (t == 4)

# select and then reshape back into (rows, hits-per-row)
out = t[mask].view(t.size(0), -1)

print(out)
# tensor([[1, 1, 1],
#         [4, 4, 4]])


tensor([[1, 1, 1],
        [4, 4, 4]])


In [29]:
t2 = t.unsqueeze(-1).broadcast_to(t.size(0), t.size(1), 2)

# mask2 = mask.unsqueeze(-1).broadcast_to(t2.size())

print(mask.shape)
print(t2.shape)

out2 = t2[mask].view(t.size(0), -1, 2)

print(out2)

torch.Size([2, 10])
torch.Size([2, 10, 2])
tensor([[[1, 1],
         [1, 1],
         [1, 1]],

        [[4, 4],
         [4, 4],
         [4, 4]]])


In [30]:
import torch

# 1) set up the toy tensor and mask, same as before
#    a→0, m→1, c→2, x→3, n→4, d→5
t = torch.tensor([
    [0,0,0, 1,1,1, 2,2,2,2],
    [3,3,3, 4,4,4, 5,5,5,5],
])
mask = (t == 1) | (t == 4)            # True for the three m’s and three n’s per row


t2 = t.unsqueeze(-1).broadcast_to(t.size(0), t.size(1), 2)
print(t2)

# 2) extract the masked elements (via masked_select + reshape)
extracted = t2[mask].view(t.size(0), -1, 2)
print("extracted:\n", extracted)
# extracted:
# tensor([[1, 1, 1],
#         [4, 4, 4]])

# 3) now scatter those back into a zero‐tensor of the original shape
dst = torch.zeros_like(t)
# flatten extracted to match the number of True’s in mask
dst = dst.masked_scatter(mask, extracted.view(-1))
print("\nscattered back:\n", dst)
# scattered back:
# tensor([[0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
#         [0, 0, 0, 4, 4, 4, 0, 0, 0, 0]])


tensor([[[0, 0],
         [0, 0],
         [0, 0],
         [1, 1],
         [1, 1],
         [1, 1],
         [2, 2],
         [2, 2],
         [2, 2],
         [2, 2]],

        [[3, 3],
         [3, 3],
         [3, 3],
         [4, 4],
         [4, 4],
         [4, 4],
         [5, 5],
         [5, 5],
         [5, 5],
         [5, 5]]])
extracted:
 tensor([[[1, 1],
         [1, 1],
         [1, 1]],

        [[4, 4],
         [4, 4],
         [4, 4]]])

scattered back:
 tensor([[0, 0, 0, 1, 1, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 0, 0, 0, 0]])


In [31]:
import torch

B, H, W = 3, 2, 5
# -------------- setup --------------


src = torch.zeros(B, H, W)
dst = torch.zeros_like(src)


src[0][0] = 1
src[0][1] = 2
src[1][0] = 3
src[1][1] = 4
src[2][0] = 5
src[2][1] = 6

dst[0][0] = 7
dst[0][1] = 8
dst[1][0] = 9
dst[1][1] = 10
dst[2][0] = 11
dst[2][1] = 12

mask = torch.zeros(B, H).bool()
mask[0][0] = True
mask[1][1] = True


scattered = src.masked_scatter(mask=mask.unsqueeze(-1), source=dst)

print("src:\n", src)
print("dst:\n", dst)
print("scattered:\n", scattered)

# inputs_embeds.masked_scatter(mask=token_dict["adv_mask"], source=adv_embed)

src:
 tensor([[[1., 1., 1., 1., 1.],
         [2., 2., 2., 2., 2.]],

        [[3., 3., 3., 3., 3.],
         [4., 4., 4., 4., 4.]],

        [[5., 5., 5., 5., 5.],
         [6., 6., 6., 6., 6.]]])
dst:
 tensor([[[ 7.,  7.,  7.,  7.,  7.],
         [ 8.,  8.,  8.,  8.,  8.]],

        [[ 9.,  9.,  9.,  9.,  9.],
         [10., 10., 10., 10., 10.]],

        [[11., 11., 11., 11., 11.],
         [12., 12., 12., 12., 12.]]])
scattered:
 tensor([[[7., 7., 7., 7., 7.],
         [2., 2., 2., 2., 2.]],

        [[3., 3., 3., 3., 3.],
         [8., 8., 8., 8., 8.]],

        [[5., 5., 5., 5., 5.],
         [6., 6., 6., 6., 6.]]])
